In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt

# 1. Parameters / paths

In [3]:
REGION = "wroclaw_small"

DATA_PATH = f"../data/{REGION}/clean.csv"

PATH_BASINS = f"../data/MPWiK_csv/geometric/Basins.csv"          
PATH_DITCHES = f"../data/MPWiK_csv/geometric/drainage_ditches.csv"   # Zmienione na CSV!
PATH_MANHOLES = f"../data/MPWiK_csv/geometric/stormwater_and_combined_manholes.csv"      
OUTPUT_SPATIAL = f"../data/{REGION}/spacial.csv"

# 2. Setting sectors

In [4]:
df = pd.read_csv(DATA_PATH)
sectors_df = df[['Sektor_ID', 'Lat', 'Lon']].drop_duplicates().copy()

geometry = [Point(lon, lat) for lon, lat in zip(sectors_df['Lon'], sectors_df['Lat'])]
gdf_sectors = gpd.GeoDataFrame(sectors_df, geometry=geometry, crs="EPSG:4326")
gdf_sectors = gdf_sectors.to_crs("EPSG:2177")

# 3. Download data

In [5]:
df_basins_raw = pd.read_csv(PATH_BASINS)
df_basins_raw['geometry'] = df_basins_raw['geometry'].apply(wkt.loads)
gdf_basins = gpd.GeoDataFrame(df_basins_raw, geometry='geometry', crs="EPSG:2177")

df_manholes_raw = pd.read_csv(PATH_MANHOLES)
df_manholes_raw['geometry'] = df_manholes_raw['geometry'].apply(wkt.loads)
gdf_manholes = gpd.GeoDataFrame(df_manholes_raw, geometry='geometry', crs="EPSG:2177")

df_ditches_raw = pd.read_csv(PATH_DITCHES)
df_ditches_raw['geometry'] = df_ditches_raw['geometry'].apply(wkt.loads)
gdf_ditches = gpd.GeoDataFrame(df_ditches_raw, geometry='geometry', crs="EPSG:2177")

# 4. Feature engineering

## - dist to ditch

In [6]:
joined_ditches = gpd.sjoin_nearest(gdf_sectors, gdf_ditches, distance_col="dist_to_ditch", how="left")
gdf_sectors['dist_to_ditch'] = joined_ditches.groupby('Sektor_ID')['dist_to_ditch'].first().values

# - manholes


In [11]:
buffered_sectors = gdf_sectors.copy()
buffered_sectors['geometry'] = buffered_sectors.geometry.buffer(300)

joined_manholes = gpd.sjoin(buffered_sectors, gdf_manholes, how="left", predicate="contains")
manhole_counts = joined_manholes.groupby('Sektor_ID')['index_right'].count().reset_index(name='manholes_300m')

gdf_sectors = gdf_sectors.merge(manhole_counts, on='Sektor_ID', how='left')

## - basins

In [14]:
joined_basins = gpd.sjoin(gdf_sectors, gdf_basins, how="left", predicate="intersects")
gdf_sectors['in_basin'] = joined_basins['index_right'].notna().astype(int).groupby(joined_basins['Sektor_ID']).first().values

# 4. Saving csv

In [12]:
final_spatial_features = pd.DataFrame(gdf_sectors.drop(columns=['geometry']))
final_spatial_features.to_csv(OUTPUT_SPATIAL, index=False)
final_spatial_features.head()

,Sektor_ID,Lat,Lon,dist_to_ditch,manholes_300m_x,in_basin,manholes_300m_y
0,S_1,51.03,16.81,9268.874786,0,0,0
1,S_10,51.11,16.83,891.480284,0,0,0
2,S_11,51.03,16.85,7536.931380,0,0,0
3,S_12,51.05,16.85,6007.602504,0,0,0
4,S_13,51.07,16.85,4330.147005,0,0,0


In [13]:
final_spatial_features.describe()

,Lat,Lon,dist_to_ditch,manholes_300m_x,in_basin,manholes_300m_y
count,30.000000,30.00000,30.000000,30.000000,30.0,30.000000
mean,51.070000,16.86000,3724.338999,5.700000,0.0,5.700000
std,0.028768,0.03474,2639.482161,24.639679,0.0,24.639679
min,51.030000,16.81000,24.503689,0.000000,0.0,0.000000
25%,51.050000,16.83000,1745.514165,0.000000,0.0,0.000000
50%,51.070000,16.86000,3481.585644,0.000000,0.0,0.000000
75%,51.090000,16.89000,5465.423965,0.000000,0.0,0.000000
max,51.110000,16.91000,9268.874786,130.000000,0.0,130.000000
